##**Aluno: Lucas Barbosa dos Santos**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip list | grep pyspark

pyspark                                  4.0.1


## Sobre os dados

O arquivo CSV contém eventos 'click' ou 'view' no tempo, de usuários em anúncios de determinadas campanhas.

**Descrição das colunas:**  
timestamp,user_id,action,adId,campaignId

**Amostra:**  
2016-09-21 22:11:00,7c74953c-66cc-48bd-9d02-a02bf039cf3f,click,adId_09,campaignId_01  
2016-06-25 18:29:00,676a083e-2f8e-4ff2-9ec2-270f7f9d6033,view,adId_09,campaignId_02  
2016-02-14 19:03:00,77158997-0dfa-48b7-9149-973dc151ef8d,click,adId_02,campaignId_02  
2016-03-26 06:27:00,78aa2467-b502-413b-94e9-04ec8210bd13,click,adId_07,campaignId_03

**Nome da pasta com os arquivos CSV:**  
data/ad_action_assignment

## Sobre as questões

Usar código de Structured Streaming na resposta.

Favor não alterar o código que gera o inputStream para manter o mesmo padrão na resposta.

Mesmo que não consiga terminar alguma questão, favor enviar, porque parte do código pode valer alguma pontuação.
## ----------------------

In [3]:
import pyspark.sql.functions as F

from pyspark.sql import SparkSession
from pyspark.sql.window import Window

In [4]:
# Criando um cluster local com 2 workers, 1 cores por worker e 3GB de RAM por worker

spark = SparkSession.builder\
    .master('local-cluster[2, 1, 3072]')\
    .getOrCreate()
spark

In [6]:
AD_ACTION_CSV_PATH = "drive/MyDrive/data 2/ad_action_assignment"

In [ ]:
# Descomente e execute para parar o SparkSession
# spark.stop()

## Foi utilizado o notebook `assignment_split_ad_action_data.ipynb` para criar os dados do streaming escrevendo arquivos CSV na pasta `data/ad_action_assignment`

## 1) Quantos eventos as campanhas geraram nos últimos 10 segundos? Ordene crescente pelo window.start, descrescente pela quantidade de eventos e calcule a cada 3 segundos. (2.5 pontos)

In [11]:
inputStream = spark.readStream.csv(
    AD_ACTION_CSV_PATH,
    schema="timestamp TIMESTAMP, \
        user_id STRING, \
        action STRING, \
        adId STRING, \
        campaignId STRING"
)

# ESCREVA SEU CÓDIGO AQUI
inputStream = inputStream \
    .groupBy(
        F.window("timestamp", "10 seconds", "3 seconds"),
        "campaignId"
    ) \
    .count() \
    .orderBy(
        F.asc("window.start"),
        F.desc("count")
    )

def foreach_batch_function(df, epoch_id):
    print(epoch_id)
    print(df.toPandas())

query = inputStream\
    .writeStream\
    .outputMode('complete')\
    .foreachBatch(foreach_batch_function)\
    .start()
query.awaitTermination(60)

0
                                        window     campaignId  count
0   (2023-09-01 03:30:51, 2023-09-01 03:31:01)  campaignId_02   1428
1   (2023-09-01 03:30:51, 2023-09-01 03:31:01)  campaignId_03   1418
2   (2023-09-01 03:30:51, 2023-09-01 03:31:01)  campaignId_01   1400
3   (2023-09-01 03:30:54, 2023-09-01 03:31:04)  campaignId_03   5923
4   (2023-09-01 03:30:54, 2023-09-01 03:31:04)  campaignId_02   5786
..                                         ...            ...    ...
67  (2023-09-01 03:31:57, 2023-09-01 03:32:07)  campaignId_03   5884
68  (2023-09-01 03:31:57, 2023-09-01 03:32:07)  campaignId_01   4918
69  (2023-09-01 03:32:00, 2023-09-01 03:32:10)  campaignId_03   1561
70  (2023-09-01 03:32:00, 2023-09-01 03:32:10)  campaignId_02   1525
71  (2023-09-01 03:32:00, 2023-09-01 03:32:10)  campaignId_01   1159

[72 rows x 3 columns]


False

In [10]:
# Stop job
query.stop()

## 2) Quais são os 5 pares de anúncio e campanha que geraram menos clicks no último minuto? Ordene crescente pelo window.start, crescente pela quantidade de clicks e calcule a cada 30 segundos. (2.5 pontos)

In [15]:
inputStream = spark.readStream.csv(
    AD_ACTION_CSV_PATH,
    schema="timestamp TIMESTAMP, \
        user_id STRING, \
        action STRING, \
        adId STRING, \
        campaignId STRING"
)

# ESCREVA SEU CÓDIGO AQUI
inputStream = inputStream \
    .where(F.col("action") == "click") \
    .groupBy(
        F.window("timestamp", "1 minute", "30 seconds"),
        "adId",
        "campaignId"
    ) \
    .count()

def foreach_batch_function(df, epoch_id):
    window = Window.partitionBy('window.start')\
        .orderBy(F.asc('count'))
    df = df.withColumn('rank', F.row_number().over(window))\
        .where(F.col('rank') <= 5)\
        .drop('rank')\
        .orderBy(F.asc('window.start'), F.asc('count'))
    print(epoch_id)
    print(df.toPandas())

query = inputStream\
    .writeStream\
    .outputMode('complete')\
    .foreachBatch(foreach_batch_function)\
    .start()
query.awaitTermination(50)

0
                                        window     adId     campaignId  count
0   (2023-09-01 03:30:30, 2023-09-01 03:31:30)  adId_03  campaignId_03   2433
1   (2023-09-01 03:30:30, 2023-09-01 03:31:30)  adId_10  campaignId_01   2460
2   (2023-09-01 03:30:30, 2023-09-01 03:31:30)  adId_04  campaignId_01   2462
3   (2023-09-01 03:30:30, 2023-09-01 03:31:30)  adId_02  campaignId_01   2505
4   (2023-09-01 03:30:30, 2023-09-01 03:31:30)  adId_07  campaignId_01   2518
5   (2023-09-01 03:31:00, 2023-09-01 03:32:00)  adId_04  campaignId_01   4950
6   (2023-09-01 03:31:00, 2023-09-01 03:32:00)  adId_02  campaignId_01   4995
7   (2023-09-01 03:31:00, 2023-09-01 03:32:00)  adId_10  campaignId_01   5010
8   (2023-09-01 03:31:00, 2023-09-01 03:32:00)  adId_01  campaignId_01   5076
9   (2023-09-01 03:31:00, 2023-09-01 03:32:00)  adId_03  campaignId_03   5109
10  (2023-09-01 03:31:30, 2023-09-01 03:32:30)  adId_04  campaignId_01   2563
11  (2023-09-01 03:31:30, 2023-09-01 03:32:30)  adId_02  campa

False

In [13]:
# Stop job
query.stop()

## 3) Qual é o total acumulado de clicks e o total acumulado de views? Calcule a medida que os dados são recebidos no streaming (2.5 pontos)

In [30]:
inputStream = spark.readStream.csv(
    AD_ACTION_CSV_PATH,
    schema="timestamp TIMESTAMP, \
        user_id STRING, \
        action STRING, \
        adId STRING, \
        campaignId STRING"
)

# ESCREVA SEU CÓDIGO AQUI
inputStream = inputStream.groupBy("action").count().alias("total")



def foreach_batch_function(df, epoch_id):
    print(epoch_id)
    print(df.toPandas())

query = inputStream\
    .writeStream\
    .outputMode('complete')\
    .foreachBatch(foreach_batch_function)\
    .start()
query.awaitTermination(35)

0
  action   count
0   view   76408
1  click  178305


False

In [28]:
# Stop job
query.stop()

## 4) Qual é a porcentagem de usuários que visualizaram e clicaram em um mesmo anúncio e campanha pelo menos uma vez nos últimos 10 segundos? Calcule a cada 10 segundos. (2.5 pontos)

Exemplo:
20% de 5 usuários clicaram e visualizaram em um mesmo anúncio e campanha pelo menos uma vez. Nesse caso, dos 5 usuários, 1 tem eventos de click e view em pelo menos 1 par de anúncio e campanha. Enquanto que, 4 usuários não apresentam esse padrão. Lembrando que cada intervalo de 10 segundos do janelamento vai ter um valor percentual associado.

```
user_id   adId	  campaignId	  action
U1		A1		C1 	 		click
U1		A1		C1 	 		view
U1		A2		C1 	 		click
U1		A2		C1 	 		view
U2		A1		C1 	 		view
U2		A2		C1 	 		click
U3		A1		C1 	 		view
U4		A2		C1 	 		click
U5		A1		C1 	 		click
```

In [34]:
inputStream = spark.readStream.csv(
    AD_ACTION_CSV_PATH,
    schema="timestamp TIMESTAMP, \
        user_id STRING, \
        action STRING, \
        adId STRING, \
        campaignId STRING"
)

# ESCREVA SEU CÓDIGO AQUI
inputStream = inputStream \
    .groupBy(
        F.window("timestamp", "10 seconds", "10 seconds"),
        "user_id",
        "adId",
        "campaignId"
    ) \
    .agg(
        F.max(F.when(F.col("action") == "click", 1).otherwise(0)).alias("has_click"),
        F.max(F.when(F.col("action") == "view", 1).otherwise(0)).alias("has_view")
    ) \
    .withColumn(
        "valid_user",
        F.when(
            (F.col("has_click") == 1) & (F.col("has_view") == 1),
            1
        ).otherwise(0)
    )

def foreach_batch_function(df, epoch_id):
    # ESCREVA SEU CÓDIGO AQUI
    # usuários válidos (que tiveram click e view)
    valid_users_df = df.where(F.col("valid_user") == 1) \
        .select("window", "user_id") \
        .distinct()

    # total de usuários por janela
    total_users_df = df.select("window", "user_id").distinct()

    valid_count = valid_users_df.groupBy("window").count() \
        .withColumnRenamed("count", "valid_users")

    total_count = total_users_df.groupBy("window").count() \
        .withColumnRenamed("count", "total_users")

    percentage_df = valid_count.join(total_count, "window") \
        .withColumn(
            "percentage",
            (F.col("valid_users") / F.col("total_users")) * 100
        ) \
        .select("window", "percentage") \
        .orderBy(F.asc("window.start"))
    print(epoch_id)
    # Garanta que esse novo DataFrame percentage_df tem as colunas window e percentage
    print(percentage_df.toPandas())

query = inputStream\
    .writeStream\
    .outputMode('complete')\
    .foreachBatch(foreach_batch_function)\
    .start()
query.awaitTermination(120)

0
                                       window  percentage
0  (2023-09-01 03:31:00, 2023-09-01 03:31:10)   64.306064
1  (2023-09-01 03:31:10, 2023-09-01 03:31:20)   64.023012
2  (2023-09-01 03:31:20, 2023-09-01 03:31:30)   61.393538
3  (2023-09-01 03:31:30, 2023-09-01 03:31:40)   63.604817
4  (2023-09-01 03:31:40, 2023-09-01 03:31:50)   64.598127
5  (2023-09-01 03:31:50, 2023-09-01 03:32:00)   64.583548
6  (2023-09-01 03:32:00, 2023-09-01 03:32:10)   20.457233


False

In [35]:
# Stop job
query.stop()